# Create CodeContests Java dataset
This notebook filters the `deepmind/code_contests` dataset for Java solutions (language code 4), de-duplicates solutions using a simple whitespace-normalization deduper, and saves JSONL and Arrow copies.

In [1]:
from datasets import load_dataset, concatenate_datasets, load_from_disk
import json
from tree_sitter import Language, Parser
import tree_sitter_java


/Users/nadimh/CodeSceneProjs/prompt-exp/py-utils/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
JAVA_LANGUAGE = Language(tree_sitter_java.language())
parser = Parser()
parser.language = JAVA_LANGUAGE


In [ ]:
code_contests = load_dataset("deepmind/code_contests")
code_contests_merged = concatenate_datasets([
  code_contests['train'],
  code_contests['test'],
  code_contests['valid']
])
code_contests_merged_filt_cols = code_contests_merged.remove_columns([
    'description', 'incorrect_solutions', 'is_description_translated', 'untranslated_description'
])


In [ ]:
def norm_node(node, src):
    t = node.type

    if t == 'include_declaration':
        return None

    if t == 'identifier':
        return ('identifier', 'VAR')

    if node.child_count == 0:
        text = src[node.start_byte:node.end_byte].decode('utf8')
        return (t, text)

    norm_children = []
    for child in node.children:
        child_norm = norm_node(child, src)
        if child_norm is not None:
            norm_children.append(child_norm)

    return (t, tuple(norm_children))

def norm_code(code: str):
    src = code.encode('utf8')
    tree = parser.parse(src)
    return norm_node(tree.root_node, src)

def dedupe_sols(sols):
    seen = set()
    unique = []
    for code in sols:
        norm = norm_code(code)
        if norm not in seen:
            seen.add(norm)
            unique.append(code)
    return unique

def comp_code_diff(code1: str, code2: str):
    norm1 = norm_code(code1)
    norm2 = norm_code(code2)
    return norm1 == norm2, norm1, norm2


In [ ]:
def filt_java_sols(row):
    new_sols = {'language': [], 'solution': []}

    all_langs = row['solutions']['language']
    all_sols = row['solutions']['solution']
    seen_sol = set()

    JAVA_LANG_CODE = 4

    for i in range(len(all_langs)):
        lang = all_langs[i]
        sol = all_sols[i]
        sol_loc = len(sol.split('\n'))

        if lang == JAVA_LANG_CODE and sol not in seen_sol and sol_loc > 60 and sol_loc < 300:
            new_sols['language'].append(lang)
            new_sols['solution'].append(sol)
            seen_sol.add(sol)

    return {'solutions': new_sols}


In [ ]:
LANGUAGE_CODES = {'UNKNOWN':0, 'PYTHON':1, 'CPP':2, 'PYTHON3':3, 'JAVA':4}

lang_code = LANGUAGE_CODES['JAVA']

java_set = code_contests_merged_filt_cols.map(
    filt_java_sols,
    batched=False
)

java_set = java_set.filter(lambda x: len(x['solutions']['language']) > 0, batched=False)

cols_to_remove = [c for c in ['input_file','output_file'] if c in java_set.column_names]

if cols_to_remove:
    java_set = java_set.remove_columns(cols_to_remove)

def keep_sols_only(row):
    if isinstance(row['solutions'], dict) and 'solution' in row['solutions']:
        return {'solutions': row['solutions']['solution']}
    return {'solutions': row['solutions']}

java_set = java_set.map(keep_sols_only, batched=False)

java_set = java_set.map(lambda x: {'solutions': dedupe_sols(x['solutions'])}, batched=False)

java_set.to_json('code_contests_java.jsonl')


print('Java dataset created')
print('Rows:', len(java_set))
print('Total solutions:', sum(len(r['solutions']) for r in java_set))

Map: 100%|██████████| 9672/9672 [00:02<00:00, 3593.10 examples/s]
Parameter 'function'=<function <lambda> at 0x1186d5ee0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.
Creating json from Arrow format: 100%|██████████| 10/10 [00:10<00:00,  1.00s/ba]


Java dataset created
Rows: 9672
Total solutions: 453189
